# Proyecto AEMET - Resumen de Extracción, Transformación y Carga (ETL) de Datos Climatológicos
Este notebook consolida y explica paso a paso todo el flujo de trabajo desarrollado para interactuar con la API de la AEMET (Agencia Estatal de Meteorología), procesar y limpiar la información, y exportar un dataset histórico listo para análisis o entrenamiento de modelos.

---

## 📋 Objetivos de este Notebook
1. **Autenticación e Integración:** Conectar con la API de AEMET OpenData utilizando una API Key almacenada de forma segura en un archivo `.env` local.
2. **Descarga del Catálogo de Estaciones:** Obtener y mostrar el maestro de estaciones meteorológicas operativas en España.
3. **Consulta de Datos de Prueba:** Descargar un subconjunto de datos diarios para una estación específica en un rango de fechas reducido para validar la conexión.
4. **Limpieza y Modelado de Datos (ETL):** Procesar la respuesta JSON de la API, reemplazando caracteres incorrectos, convirtiendo tipos de datos (strings a floats con coma, datetime) y manejando nulos de forma robusta.
5. **Descarga Masiva e Histórica:** Implementar un bucle para extraer datos históricos de los últimos años, gestionando de forma proactiva los límites de velocidad (rate limiting) de la API mediante retardos estratégicos.
6. **Exportación de Datos:** Consolidar el dataset completo en archivos Excel y CSV en el espacio local del usuario.
7. **Visualización Resumen:** Crear un gráfico de control que muestre la evolución del clima a lo largo del tiempo para validar los datos visualmente.

In [ ]:
# Imports de librerías esenciales
import os
import sys
import time
import json
import calendar
import subprocess
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv

# Comprobación e instalación automática de dependencias críticas de terceros
def asegurar_libreria(nombre_paquete, nombre_import=None):
    if nombre_import is None:
        nombre_import = nombre_paquete
    try:
        __import__(nombre_import)
    except ImportError:
        print(f"Librería '{nombre_paquete}' no encontrada. Instalando automáticamente...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", nombre_paquete])
        print(f"Librería '{nombre_paquete}' instalada y cargada.")

asegurar_libreria("openpyxl")
asegurar_libreria("matplotlib", "matplotlib.pyplot")
asegurar_libreria("seaborn")

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

print("Librerías importadas y configuradas correctamente.")

## Paso 1: Configuración del Entorno y Carga de Variables Seguras

Para evitar exponer credenciales en el código fuente, la API Key de AEMET se almacena en el archivo `.env` de esta carpeta. Usamos la biblioteca `python-dotenv` para cargar el entorno de forma segura, apuntando específicamente al directorio de este notebook.

In [ ]:
# Definir la ruta del archivo .env de Miguel
BASE_DIR = Path().resolve()
env_path = BASE_DIR / '.env'

print(f"Buscando archivo de variables de entorno en: {env_path}")

if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print("Archivo .env cargado con éxito.")
else:
    print("ADVERTENCIA: No se encontró el archivo .env en el directorio actual.")
    print("Se intentará cargar el entorno general de la sesión (o del directorio padre).")
    load_dotenv()

# Obtener y validar la API Key
API_KEY = os.getenv('AEMET_API_KEY')
if not API_KEY:
    raise RuntimeError('ERROR CRÍTICO: La variable AEMET_API_KEY no está definida en el archivo .env')
else:
    # Mostramos los primeros y últimos caracteres por seguridad
    print(f"API Key detectada correctamente: {API_KEY[:10]}...{API_KEY[-10:]}")

## Paso 2: El Protocolo de Conexión de AEMET OpenData

La API de AEMET funciona bajo un modelo de **"doble llamada"**:
1. Enviamos nuestra consulta con la clave `api_key` en las cabeceras (headers) de la solicitud HTTP GET.
2. AEMET valida la solicitud y nos responde con un JSON de metadatos que contiene una URL temporal en el campo `datos`.
3. Hacemos una segunda solicitud `GET` a esa URL temporal (esta vez sin cabeceras de autorización) para descargar los datos reales del sensor en formato JSON.

Definimos el diccionario de cabeceras que utilizaremos en la primera llamada de autenticación.

In [ ]:
# Configuración de cabeceras de autenticación
headers = {
    'cache-control': "no-cache",
    'api_key': API_KEY,
    'accept': "application/json"
}

print("Cabeceras configuradas correctamente.")

## Paso 3: Obtener el Inventario Completo de Estaciones Climatológicas

Para realizar cualquier consulta histórica, necesitamos conocer el identificador único (`indicativo` o `idema`) de la estación meteorológica. El endpoint `/api/valores/climatologicos/inventarioestaciones/todasestaciones` nos devuelve el catálogo completo de las estaciones activas en España.

In [ ]:
def obtener_inventario_estaciones():
    url_base = "https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/todasestaciones"
    
    print(f"[API AEMET] Consultando inventario en: {url_base}")
    try:
        response = requests.get(url_base, headers=headers)
        if response.status_code == 200:
            meta_datos = response.json()
            print(f"Respuesta de control AEMET: {meta_datos.get('descripcion')} (Estado: {meta_datos.get('estado')})")
            
            if meta_datos.get('estado') == 200:
                url_datos = meta_datos.get('datos')
                print("   -> Enlace temporal de datos obtenido con éxito.")
                
                # Segunda llamada: Descarga del JSON real
                datos_response = requests.get(url_datos)
                if datos_response.status_code == 200:
                    return datos_response.json()
                else:
                    print(f"Error al descargar los datos finales: {datos_response.status_code}")
            else:
                print(f"Error reportado por AEMET: {meta_datos.get('descripcion')}")
        else:
            print(f"Error de red en la primera llamada: HTTP {response.status_code}")
    except Exception as e:
        print(f"Excepción ocurrida: {e}")
    return None

# Ejecutar y mostrar una muestra del catálogo
estaciones = obtener_inventario_estaciones()

if estaciones:
    print(f"\nSe han obtenido {len(estaciones)} estaciones meteorológicas.")
    df_estaciones = pd.DataFrame(estaciones)
    print("\nMuestra de las primeras 5 estaciones en el inventario:")
    display(df_estaciones.head(5))
else:
    print("No se pudo obtener el inventario de estaciones. Verifique su API Key.")

## Paso 4: Consulta de Prueba de Valores Climatológicos Diarios

Para verificar el flujo de descarga de datos de sensores diarios, consultaremos los datos del 1 al 15 de mayo de 2026 para la estación de **Palma de Mallorca (Puerto)** (indicativo o idema: `"B228"`).

El formato del endpoint requerido es:
`/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini}/fechafin/{fecha_fin}/estacion/{idema}`

In [ ]:
def obtener_valores_climaticos(fecha_ini_utc, fecha_fin_utc, idema):
    url_base = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini_utc}/fechafin/{fecha_fin_utc}/estacion/{idema}"
    
    try:
        response = requests.get(url_base, headers=headers)
        if response.status_code == 200:
            meta_datos = response.json()
            if meta_datos.get('estado') == 200:
                url_final = meta_datos.get('datos')
                datos_response = requests.get(url_final)
                if datos_response.status_code == 200:
                    return datos_response.json()
                else:
                    print(f"Error al descargar los datos meteorológicos: {datos_response.status_code}")
            else:
                return None
        else:
            print(f"Error de red: HTTP {response.status_code}")
    except Exception as e:
        print(f"Excepción en la consulta climatológica: {e}")
    return None


def obtener_valores_climaticos_todas(fecha_ini_utc, fecha_fin_utc, max_retries=15, base_delay=5.0):
    url_base = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini_utc}/fechafin/{fecha_fin_utc}/todasestaciones"
    
    delay = base_delay
    for attempt in range(max_retries):
        try:
            response = requests.get(url_base, headers=headers)
            
            # Si es un error de autenticación o recurso no encontrado, no reintentamos
            if response.status_code in [401, 403, 404]:
                print(f"   [Error Fatal] HTTP {response.status_code}. No se reintentará.")
                return None
                
            if response.status_code == 200:
                meta_datos = response.json()
                estado = meta_datos.get('estado')
                
                if estado == 200:
                    url_final = meta_datos.get('datos')
                    datos_response = requests.get(url_final)
                    if datos_response.status_code == 200:
                        return datos_response.json()
                    elif datos_response.status_code in [401, 403, 404]:
                        print(f"   [Error Fatal en datos] HTTP {datos_response.status_code}. No se reintentará.")
                        return None
                    else:
                        print(f"   [Intento {attempt+1}/{max_retries}] Error al descargar los datos meteorológicos: {datos_response.status_code}")
                elif estado in [401, 403, 404]:
                    print(f"   [Error Fatal API] Estado {estado}: {meta_datos.get('descripcion')}. No se reintentará.")
                    return None
                elif estado == 429:
                    print(f"   [Intento {attempt+1}/{max_retries}] API AEMET retornó Estado 429 (Límite superado).")
                else:
                    print(f"   [Intento {attempt+1}/{max_retries}] API AEMET retornó Estado {estado}: {meta_datos.get('descripcion')}")
            elif response.status_code == 429:
                print(f"   [Intento {attempt+1}/{max_retries}] HTTP 429 (Too Many Requests).")
            else:
                print(f"   [Intento {attempt+1}/{max_retries}] HTTP {response.status_code}")
                
        except Exception as e:
            print(f"   [Intento {attempt+1}/{max_retries}] Excepción en conexión/petición: {e}")
            
        if attempt < max_retries - 1:
            print(f"      ⚠️ Pausando {delay} segundos antes de reintentar...")
            time.sleep(delay)
            delay *= 2.0
            
    print(f"   ❌ Se agotaron los {max_retries} intentos de descarga para este lote.")
    return None

# Parámetros de prueba
idema_prueba = "B228"
ini_prueba = "2026-05-01T00:00:00UTC"
fin_prueba = "2026-05-15T00:00:00UTC"

print("Ejecutando consulta de prueba de datos meteorológicos...")
climaticos_prueba = obtener_valores_climaticos(ini_prueba, fin_prueba, idema_prueba)

if climaticos_prueba:
    print(f"¡Éxito! Se obtuvieron {len(climaticos_prueba)} registros diarios.")
    print("Muestra en crudo del primer registro (JSON):")
    print(json.dumps(climaticos_prueba[0], indent=2, ensure_ascii=False))
else:
    print("No se obtuvieron datos. Verifique que las fechas sean correctas y la API Key sea válida.")

## Paso 5: Procesamiento y Limpieza de Datos (Pipeline ETL)

Los datos crudos de AEMET vienen completamente en formato de texto (strings). Además, la representación decimal utiliza la coma española (`,`) en lugar del punto (`.`), lo cual impide operaciones aritméticas en Pandas.

Implementamos la función `limpiar_y_cargar_datos` que realiza el siguiente flujo de transformación:
1. Extrae los campos de interés de cada registro.
2. Convierte el campo `fecha` a tipo `datetime` de Pandas.
3. Normaliza las representaciones numéricas: reemplaza las comas por puntos en variables continuas y las convierte a tipo `float32` o `int` de forma segura.
4. Convierte las horas de temperaturas/presiones máximas y mínimas a objetos `datetime.time`, gestionando de forma robusta valores de texto anómalos como 'Varias' o nulos.
5. Limpia el campo de precipitación (`prec`), convirtiendo el valor especial "Ip" (inapreciable) a `0.0` para poder realizar cálculos y sumas acumuladas consistentes.

In [ ]:
def limpiar_y_cargar_datos(datos_json):
    if not datos_json:
        return pd.DataFrame()
        
    data = []
    for dia in datos_json:
        data.append({
            "fecha": dia.get("fecha"),
            "indicativo": dia.get("indicativo"),
            "nombre": dia.get("nombre"),
            "provincia": dia.get("provincia"),
            "altitud": dia.get("altitud"),
            "tmed": dia.get("tmed"),
            "prec": dia.get("prec", "0,00"),
            "tmin": dia.get("tmin"),
            "horatmin": dia.get("horatmin"),
            "tmax": dia.get("tmax"),
            "horatmax": dia.get("horatmax"),
            "dir": dia.get("dir"),
            "velmedia": dia.get("velmedia"),
            "racha": dia.get("racha"),
            "horaracha": dia.get("horaracha"),
            "sol": dia.get("sol"),
            "presMax": dia.get("presMax"),
            "horaPresMax": dia.get("horaPresMax"),
            "presMin": dia.get("presMin"),
            "horaPresMin": dia.get("horaPresMin"),
            "hrMedia": dia.get("hrMedia"),
            "hrMax": dia.get("hrMax"),
            "horaHrMax": dia.get("horaHrMax"),
            "hrMin": dia.get("hrMin"),
            "horaHrMin": dia.get("horaHrMin")
        })

    df = pd.DataFrame(data)

    # 1. Normalización de campos de texto y fecha
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["indicativo"] = df["indicativo"].astype(str)
    df["nombre"] = df["nombre"].astype(str)
    df["provincia"] = df["provincia"].astype(str)
    df["altitud"] = pd.to_numeric(df["altitud"], errors="coerce")

    # Función auxiliar para convertir decimales con coma a punto
    def a_float(serie):
        if serie is None:
            return np.nan
        return serie.astype(str).str.replace(",", ".").replace("", "nan").astype(np.float32)

    # 2. Conversión de variables continuas
    float_cols = ["tmed", "tmin", "tmax", "velmedia", "racha", "sol", "presMax", "presMin", "hrMedia", "hrMax", "hrMin"]
    for col in float_cols:
        if col in df.columns:
            df[col] = a_float(df[col])

    # 3. Tratamiento especial de la precipitación (Ip = Inapreciable = 0.0 mm)
    if "prec" in df.columns:
        df["prec"] = df["prec"].astype(str).str.replace("Ip", "0.0").str.replace(",", ".").replace("", "0.0")
        df["prec"] = pd.to_numeric(df["prec"], errors="coerce").astype(np.float32)

    # 4. Normalización de horas a objetos datetime.time
    time_cols = ["horatmin", "horatmax", "horaHrMax", "horaHrMin"]
    for col in time_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).replace(["Varias", "nan", "None", ""], np.nan)
            df[col] = pd.to_datetime(df[col], format="%H:%M", errors="coerce").dt.time

    # 5. Normalización de horas de presión (formato HH)
    hour_cols = ["horaPresMax", "horaPresMin"]
    for col in hour_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).replace(["Varias", "nan", "None", ""], np.nan)
            df[col] = pd.to_datetime(df[col], format="%H", errors="coerce").dt.time

    if "dir" in df.columns:
        df["dir"] = df["dir"].astype(str)
    if "horaracha" in df.columns:
        df["horaracha"] = df["horaracha"].astype(str)

    return df

# Validar el pipeline con el dataset de prueba
df_prueba = limpiar_y_cargar_datos(climaticos_prueba)
print("¡Transformación y tipado completados con éxito!")
print("\nTipos de datos del DataFrame procesado:")
df_prueba.info()
print("\nMuestra del DataFrame final procesado:")
display(df_prueba.head(5))

## Paso 6: Extracción Histórica Masiva con Control de Rate Limiting

Para obtener un dataset consistente e histórico, se implementa un bucle recursivo que descarga datos año por año y mes por mes.

**⚠️ IMPORTANTE - Límites de la API de AEMET:**
AEMET limita estrictamente la cantidad de llamadas simultáneas. Si no se manejan pausas, la API responderá con errores `HTTP 429` (Demasiadas peticiones) o bloqueará el token de usuario.
Para evitar esto, implementamos:
1. **Pausa de 60 segundos** entre cada año consultado.
2. **Pausa de 1.5 segundos** entre cada mes consultado.
3. **Pausa de 40 minutos** de seguridad si el tiempo total de ejecución supera los 5 minutos de forma continua.
4. **Saltado dinámico de meses futuros** para evitar llamadas inútiles sobre periodos que aún no han ocurrido.

*Nota de control:* Por defecto, esta descarga está pre-configurada para extraer los **últimos 3 años** (2024-2026) para optimizar el tiempo de prueba de este notebook. Si deseas obtener la serie histórica completa de los últimos 10 años, simplemente modifica la variable `AÑOS_A_CONSULTAR = 10`.

In [ ]:
from datetime import datetime, timedelta
import gc

# Configuración del histórico a descargar (todas las estaciones)
fecha_fin_global = datetime(2026, 5, 30)
DIAS_A_CONSULTAR = 3650  # 10 años para descargar completo
fecha_inicio_global = fecha_fin_global - timedelta(days=DIAS_A_CONSULTAR - 1)

OUTPUT_CSV_DIR = BASE_DIR / 'sheets' / 'csv'
OUTPUT_XLSX_DIR = BASE_DIR / 'sheets' / 'xlsx'
OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_XLSX_DIR.mkdir(parents=True, exist_ok=True)

año_inicio = fecha_inicio_global.year
año_fin = fecha_fin_global.year

print(f"🚀 Iniciando proceso de descarga masiva y exportación anual desde {año_inicio} hasta {año_fin}...")

for anio in range(año_fin, año_inicio - 1, -1):
    print(f"\n=========================================")
    print(f"⏳ PROCESANDO AÑO CLIMATOLÓGICO: {anio}")
    print(f"=========================================")
    
    ini_anio = max(datetime(anio, 1, 1), fecha_inicio_global)
    fin_anio = min(datetime(anio, 12, 31), fecha_fin_global)
    
    df_anio = pd.DataFrame()
    fecha_actual = fin_anio
    
    lote_size = 15
    dias_anio_procesados = 0
    total_dias_anio = (fin_anio - ini_anio).days + 1
    
    print(f"Rango para {anio}: desde {ini_anio.strftime('%Y-%m-%d')} hasta {fin_anio.strftime('%Y-%m-%d')} ({total_dias_anio} días)")
    
    while dias_anio_procesados < total_dias_anio:
        dias_restantes = total_dias_anio - dias_anio_procesados
        dias_lote = min(lote_size, dias_restantes)
        
        fecha_ini_lote = fecha_actual - timedelta(days=dias_lote - 1)
        
        fecha_ini_str = fecha_ini_lote.strftime("%Y-%m-%dT00:00:00UTC")
        fecha_fin_str = fecha_actual.strftime("%Y-%m-%dT23:59:59UTC")
        
        print(f"\n🚀 Iniciando descarga de todas las estaciones desde {fecha_ini_str} hasta {fecha_fin_str}...")
        
        # Pausa estándar de 2 segundos entre descargas exitosas para mitigar rate limiting
        print("Esperando 2 segundos para mantener tasa estándar...")
        time.sleep(2.0)
        
        datos_json = obtener_valores_climaticos_todas(fecha_ini_str, fecha_fin_str)
        
        if datos_json:
            df_lote = limpiar_y_cargar_datos(datos_json)
            df_anio = pd.concat([df_anio, df_lote], ignore_index=True)
            print(f"   ✅ Lote procesado con éxito. Registros agregados en lote: {len(df_lote)}")
        else:
            print(f"   ❌ No se pudieron descargar datos para el lote {fecha_ini_str} a {fecha_fin_str}")
            
        dias_anio_procesados += dias_lote
        fecha_actual = fecha_ini_lote - timedelta(days=1)
        
    # Guardar y limpiar memoria para el año procesado
    if not df_anio.empty:
        # Ordenar por fecha cronológica ascendente
        if 'fecha' in df_anio.columns:
            df_anio = df_anio.sort_values(by='fecha', ascending=True)
            
        csv_path = OUTPUT_CSV_DIR / f'climaticos_{anio}.csv'
        excel_path = OUTPUT_XLSX_DIR / f'climaticos_{anio}.xlsx'
        
        print(f"\n💾 Guardando archivos físicos del año {anio}...")
        try:
            # Exportar a CSV
            df_anio.to_csv(csv_path, index=False)
            print(f"   [OK] Dataset CSV guardado en: {csv_path}")
            
            # Exportar a Excel
            df_anio.to_excel(excel_path, index=False)
            print(f"   [OK] Dataset Excel guardado en: {excel_path}")
        except Exception as e:
            print(f"   ❌ Error al exportar los archivos del año {anio}: {e}")
            
        # Liberar memoria explícitamente
        del df_anio
        gc.collect()
    else:
        print(f"⚠️ No se obtuvieron datos para el año {anio}, saltando exportación.")

print("\n🎉 ¡Proceso completo de descarga masiva y exportación segmentada finalizado!")


## Paso 7: Verificación de los Archivos Físicos Generados

Para validar que la exportación anual y segmentada se ha completado correctamente en disco, en esta sección listamos el contenido de los directorios de salida  y , mostrando el tamaño ocupado por cada archivo generado.

In [ ]:
# Verificación de los archivos exportados
OUTPUT_CSV_DIR = BASE_DIR / 'sheets' / 'csv'
OUTPUT_XLSX_DIR = BASE_DIR / 'sheets' / 'xlsx'

print("Archivos CSV generados:")
if OUTPUT_CSV_DIR.exists():
    for arch in sorted(OUTPUT_CSV_DIR.glob('climaticos_*')):
        print(f" - csv/{arch.name} ({arch.stat().st_size / (1024 * 1024):.2f} MB)")
else:
    print("El directorio sheets/csv/ no existe.")

print("\nArchivos Excel generados:")
if OUTPUT_XLSX_DIR.exists():
    for arch in sorted(OUTPUT_XLSX_DIR.glob('climaticos_*')):
        print(f" - xlsx/{arch.name} ({arch.stat().st_size / (1024 * 1024):.2f} MB)")
else:
    print("El directorio sheets/xlsx/ no existe.")

## ℹ️ Nota sobre la Variabilidad en el Número de Estaciones (Idemas) por Lote

Es común observar que, aunque el inventario maestro de AEMET reporta aproximadamente **920 estaciones activas**, el número de registros diarios reales por lote no equivale exactamente a `920 estaciones × N días`.

Por ejemplo, en un lote de 15 días (como del 16 al 30 de mayo), se obtienen unos **12,176 registros**, lo que equivale a un promedio de **~812 estaciones diarias** con reporte de datos, en lugar de las 920 teóricas.

### ¿A qué se debe esta diferencia?
1. **Estaciones inactivas o en mantenimiento:** Algunas estaciones pueden sufrir fallos de comunicación, averías en sensores o estar en periodos de mantenimiento programado.
2. **Validación y desfase temporal de datos:** AEMET somete los datos climatológicos a controles de calidad antes de publicarlos en la API OpenData. Las estaciones de publicación reciente o secundarias pueden tardar más en volcar sus datos.
3. **Altas y bajas en la red:** El inventario incluye estaciones históricas o de alta reciente que no necesariamente reportan en la horquilla temporal consultada.
4. **Filtros de datos nulos:** Durante el proceso de ingesta, aquellos registros que no contienen lecturas válidas o están totalmente vacíos para los días del lote son descartados o no transmitidos por la propia API.